# Treinamento com intrusion_gap

Fluxo replicando o notebook `model_training` para o dataset `intrusion_gap`, focando em seleção de características (mesmos 5 métodos) e avaliação de modelos em múltiplas estratégias de cross-validation estratificada para lidar com o forte desbalanceamento (1898 intrusão vs 78995 normal).

In [ ]:
# Imports principais e configuração de ambiente
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from collections import defaultdict

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedShuffleSplit,
    RepeatedStratifiedKFold,
    train_test_split,
    cross_validate,
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
)
from sklearn.metrics import confusion_matrix, matthews_corrcoef
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Verifica disponibilidade do mrmr-selection
try:
    from mrmr import mrmr_classif
    MRMR_AVAILABLE = True
except Exception:
    MRMR_AVAILABLE = False

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)
sns.set_style("whitegrid")

In [ ]:
# Carrega a base intrusion_gap e prepara alvo
base_path = Path("../data/processed")
data_path = base_path / "intrusion_gap.csv"

df = pd.read_csv(data_path)
print(f"Linhas x colunas: {df.shape}")
print(df["type"].value_counts())

# Encode do alvo (normal = 0, intrusão = 1)
target_col = "type"
le = LabelEncoder()
df[target_col] = le.fit_transform(df[target_col])

# Seleciona apenas colunas numéricas (evita strings como MAC/IP)
num_df = df.select_dtypes(include=[np.number]).copy()

# Remove colunas puramente constantes para evitar ruído
constant_cols = [c for c in num_df.columns if num_df[c].nunique() <= 1]
if constant_cols:
    num_df = num_df.drop(columns=constant_cols)

# Trata infinitos e faltantes
num_df = num_df.replace([np.inf, -np.inf], 0).fillna(0)

X = num_df.drop(columns=[target_col])
y = num_df[target_col].values

print(f"Total de atributos numéricos: {X.shape[1]}")
print(f"Classe positiva: {np.sum(y==1)} | Classe negativa: {np.sum(y==0)}")

In [ ]:
# Funções auxiliares de seleção de características
K = 15  # quantidade padrão de atributos a manter por método


def select_mrmr(X_df, y_vec, k=K):
    """Seleciona k features pelo critério mRMR se disponível."""
    if not MRMR_AVAILABLE:
        return []
    return list(mrmr_classif(X=X_df, y=y_vec, K=k))


def select_fisher_kbest(X_df, y_vec, k=K):
    selector = SelectKBest(score_func=f_classif, k=min(k, X_df.shape[1]))
    selector.fit(X_df, y_vec)
    return list(X_df.columns[selector.get_support()])


def fisher_score_manual(X_df, y_vec, k=K):
    # Razão entre variância entre classes e intra-classes
    scores = {}
    classes = np.unique(y_vec)
    for col in X_df.columns:
        between = 0.0
        within = 0.0
        overall_mean = X_df[col].mean()
        for cls in classes:
            cls_vals = X_df.loc[y_vec == cls, col]
            cls_mean = cls_vals.mean()
            between += len(cls_vals) * (cls_mean - overall_mean) ** 2
            within += len(cls_vals) * cls_vals.var()
        scores[col] = between / (within + 1e-8)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [feat for feat, _ in ranked[:k]]


def select_pearson(X_df, y_vec, k=K):
    corr = X_df.apply(lambda col: np.abs(np.corrcoef(col, y_vec)[0, 1]))
    corr = corr.replace([np.inf, -np.inf], 0).fillna(0)
    ranked = corr.sort_values(ascending=False)
    return list(ranked.head(min(k, len(ranked))).index)


def select_extratrees(X_df, y_vec, k=K):
    model = ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced")
    model.fit(X_df, y_vec)
    importances = pd.Series(model.feature_importances_, index=X_df.columns)
    ranked = importances.sort_values(ascending=False)
    return list(ranked.head(min(k, len(ranked))).index)


def select_linsvc(X_df, y_vec, k=K):
    svc = LinearSVC(C=0.1, penalty="l1", dual=False, random_state=42)
    svc.fit(StandardScaler().fit_transform(X_df), y_vec)
    coefs = np.abs(svc.coef_[0]) if svc.coef_.ndim == 2 else np.abs(svc.coef_)
    coef_series = pd.Series(coefs, index=X_df.columns)
    ranked = coef_series.sort_values(ascending=False)
    return list(ranked.head(min(k, len(ranked))).index)


def select_lasso(X_df, y_vec, k=K):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_df)
    lasso = LassoCV(cv=5, random_state=42, n_jobs=-1)
    lasso.fit(X_scaled, y_vec)
    coefs = pd.Series(np.abs(lasso.coef_), index=X_df.columns)
    ranked = coefs.sort_values(ascending=False)
    return list(ranked.head(min(k, len(ranked))).index)


def select_low_variance(X_df, threshold=0.05):
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(X_df)
    return list(X_df.columns[selector.get_support()])


def summarize_feature_sets(feature_sets):
    rows = []
    for method, feats in feature_sets.items():
        for rank, feat in enumerate(feats, 1):
            rows.append({"method": method, "feature": feat, "rank": rank})
    return pd.DataFrame(rows)

In [ ]:
# Executa os seletores (mesmos 5 métodos do notebook base + variância)
feature_sets = {
    "mrmr": select_mrmr(X, y),
    "fisher_kbest": select_fisher_kbest(X, y),
    "fisher_manual": fisher_score_manual(X, y),
    "pearson": select_pearson(X, y),
    "extratrees": select_extratrees(X, y),
    "linsvc_l1": select_linsvc(X, y),
    "lasso": select_lasso(X, y),
    "low_variance": select_low_variance(X),
}

# Remove seleções vazias (caso mRMR não esteja instalado)
feature_sets = {k: v for k, v in feature_sets.items() if len(v) > 0}

summary_df = summarize_feature_sets(feature_sets)
print("Métodos disponíveis:", list(feature_sets.keys()))
summary_df.head()

In [ ]:
# Visualiza frequência de seleção por método
if not summary_df.empty:
    freq = summary_df.groupby("feature").size().sort_values(ascending=False).head(20)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=freq.values, y=freq.index, palette="viridis")
    plt.title("Top 20 features mais frequentes entre os métodos")
    plt.xlabel("Contagem de ocorrência")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum conjunto de features disponível para plot.")

In [ ]:
# Define modelos e estratégias de cross-validation (5 modos diferentes)
class_weights = {0: 1.0, 1: max(1.0, (len(y) - np.sum(y)) / max(np.sum(y), 1))}

model_factories = {
    "rf": lambda: RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        random_state=42,
        class_weight=class_weights,
    ),
    "logreg": lambda: Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=200,
                    class_weight="balanced",
                    solver="lbfgs",
                    n_jobs=-1,
                ),
            ),
        ]
    ),
    "svc_rbf": lambda: Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                SVC(
                    probability=True,
                    class_weight="balanced",
                    kernel="rbf",
                    gamma="scale",
                    random_state=42,
                ),
            ),
        ]
    ),
    "knn": lambda: Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=7))]),
    "nb": lambda: GaussianNB(),
    "dt": lambda: DecisionTreeClassifier(max_depth=None, random_state=42, class_weight=class_weights),
}

cv_strategies = {
    "StratKFold-5": StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    "StratKFold-10": StratifiedKFold(n_splits=10, shuffle=True, random_state=42),
    "StratShuffle-20": StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=42),
    "StratShuffle-10": StratifiedShuffleSplit(n_splits=5, test_size=0.1, random_state=42),
    "RepeatedStrat-5x2": RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=42),
}

In [ ]:
# Função de avaliação cruzada por conjunto de features e estratégia de CV
scoring = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "balanced_accuracy": "balanced_accuracy",
    "roc_auc": "roc_auc_ovr",
}


def evaluate_feature_sets_cv(X_df, y_vec, feature_sets, model_factories, cv_strategies):
    results = []
    for feat_name, feat_list in feature_sets.items():
        X_sel = X_df[feat_list]
        for model_name, factory in model_factories.items():
            for cv_name, cv in cv_strategies.items():
                model = factory()
                cv_res = cross_validate(
                    model,
                    X_sel,
                    y_vec,
                    cv=cv,
                    scoring=scoring,
                    n_jobs=-1,
                    return_estimator=False,
                )
                for metric, values in cv_res.items():
                    if not metric.startswith("test_"):
                        continue
                    metric_name = metric.replace("test_", "")
                    results.append(
                        {
                            "feature_set": feat_name,
                            "model": model_name,
                            "cv": cv_name,
                            "metric": metric_name,
                            "mean": np.mean(values),
                            "std": np.std(values),
                        }
                    )
    return pd.DataFrame(results)


results_df = evaluate_feature_sets_cv(X, y, feature_sets, model_factories, cv_strategies)
results_df.head()

In [ ]:
# Ranking dos melhores modelos por conjunto de features usando F1 macro
if not results_df.empty:
    f1_df = results_df[results_df["metric"] == "f1_macro"].copy()
    best_per_set = (
        f1_df.sort_values(["feature_set", "mean"], ascending=[True, False])
        .groupby(["feature_set", "cv"], as_index=False)
        .first()
    )
    display(best_per_set.head(20))
else:
    print("Execute a célula anterior para obter resultados de CV.")

In [ ]:
# Plot comparativo: F1 macro médio por feature set e modelo (melhor CV de cada um)
if not results_df.empty:
    plot_df = results_df[results_df["metric"] == "f1_macro"].copy()
    # Seleciona a melhor combinação por feature_set + model (maior média entre CVs)
    plot_df = plot_df.sort_values(["feature_set", "model", "mean"], ascending=[True, True, False])
    plot_df = plot_df.groupby(["feature_set", "model"], as_index=False).first()

    plt.figure(figsize=(12, 6))
    sns.barplot(data=plot_df, x="mean", y="feature_set", hue="model", palette="tab20")
    plt.xlabel("F1 macro (média)")
    plt.ylabel("Conjunto de features")
    plt.title("Comparação de F1 macro por seleção de características e modelo")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum resultado para plotar. Execute as avaliações de CV primeiro.")

In [ ]:
# Avaliação simples hold-out no melhor conjunto (primeiro da lista) para gerar matrizes de confusão/ROC
if feature_sets:
    best_set_name = list(feature_sets.keys())[0]
    best_feats = feature_sets[best_set_name]
    X_sel = X[best_feats]

    X_train, X_test, y_train, y_test = train_test_split(
        X_sel, y, test_size=0.2, random_state=42, stratify=y
    )

    rf = RandomForestClassifier(
        n_estimators=300, random_state=42, n_jobs=-1, class_weight=class_weights
    )
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    y_proba = rf.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    bal = balanced_accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    print(f"Hold-out | set={best_set_name} | acc={acc:.3f} | f1={f1:.3f} | bal_acc={bal:.3f} | roc_auc={roc:.3f}")

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.title(f"Matriz de confusão - {best_set_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum conjunto de features disponível para avaliação hold-out.")

## Próximos passos
- Rodar as células de seleção e avaliação para comparar os 5 modos de cross-validation.
- Revisar o melhor par `feature_set` + `modelo` olhando F1 macro e balanced accuracy.
- Se necessário, ajustar hiperparâmetros dos modelos campeões e repetir os gráficos.